# Suite TS5DEF — TypeScript 5 definitions, read from the store

The names a TypeScript 5 program declares, in the scopes ECMAScript and TypeScript give them
(`TypeScript5.Definitions.define(store, path)`), read from the program the store holds and computed on demand, nothing
stored: one entity per name and kind per scope, merging what TypeScript merges, and lookup outward that keeps values,
types and namespaces apart. A place is an `Anchor`, plain data that outlives the transaction it was read in: its
statement's place below the program, the statement's property and the term's path within it, written as text
(`body[1].consequent.body[0]:expression.left`), by which each case names the places it checks. The whole tree is
values, statements within terms included: a function expression's or an arrow's statements are its term's `block`,
a class expression's members its `classBody`, each anchored at the term's place followed by the statement places within
(`body[0]:declarations[0].init.block.body[1]`). Each case parses a program into one store (`TypeScript5.Parse.parse`),
defines it, and checks the entities, their declarations, and what names refer to. The cases port TSDEF's.

In [ ]:
import itertools
import time
from pathlib import Path

from mbse.Expressions.Framework import Objects, Terms
from mbse.Expressions.TypeScript.TypeScript5 import Expressions as TypeScriptExpressions
from mbse.Programs import TypeScript5
from mbse.Programs.Framework.Model import WORKSPACE
from mbse.Programs.TypeScript5 import Definitions as D, Parse, Statements
from mbse.Schemas.Framework import Schemas
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = TypeScript5.store()
PATHS = (f"case{n}" for n in itertools.count())


def define(text, extension):
    """The program `text` parses to, written in the store at a path of its own ending `extension`, which decides
    whether JSX is read."""
    path = next(PATHS) + extension
    STORE.transaction(Parse.parse(text, path))
    return D.define(STORE, path)


def listing(program):
    return [(e.kind, e.qualified_name(), len(e.declarations)) for e in program.entities()]


def refers(program, at):
    return [f"{e.kind}:{e.qualified_name()}" for e in D.referents(program, at)]


def named(program, spelling):
    """The places a name spelled `spelling` is written, as text, in source order."""
    return [str(at) for at in program.named(spelling)]

## TS5DEF-01 · What declarations declare, and which merge: one entity per name and kind per scope

In [ ]:
program = define("""
var v = 1; var v;
let [a, , ...b] = c, { d, e: f = 1, ...g } = h;
const k = 1;
function over(x: string): string;
function over(x: any) { return x; }
interface I { a: string }
interface I { b: number }
namespace N { export const x = 1; }
namespace N { export const y = 2; }
enum E { A }
enum E { B = 1 }
type T = string;
class C {}
import def, { one as two, type three } from "m";
import * as all from "m";
import eq = require("m");
declare function amb(): void;
export default function () {}
""", ".ts")
assert listing(program) == [
    ("variable", "v", 2), ("variable", "a", 1), ("variable", "b", 1), ("variable", "d", 1), ("variable", "f", 1),
    ("variable", "g", 1), ("variable", "k", 1), ("function", "over", 2), ("parameter", "over.x", 1),
    ("parameter", "over.x", 1), ("interface", "I", 2), ("property", "I.a", 1), ("property", "I.b", 1),
    ("namespace", "N", 2), ("variable", "N.x", 1), ("variable", "N.y", 1), ("enum", "E", 2),
    ("enum member", "E.A", 1), ("enum member", "E.B", 1), ("type alias", "T", 1), ("class", "C", 1),
    ("import", "def", 1), ("import", "two", 1), ("import", "three", 1), ("import", "all", 1), ("import", "eq", 1),
    ("function", "amb", 1)]
assert program.lookup("v")[0].declarations == ["body[0]:declarations[0].id", "body[1]:declarations[0].id"]
assert program.lookup("b")[0].definition == "body[2]:declarations[0].id.elements[2].argument"  # `...b`
assert program.lookup("f")[0].definition == "body[2]:declarations[1].id.properties[1].value.left"  # `e: f = 1`
over = program.lookup("over")[0]
assert over.declarations == ["body[4]:id", "body[5]:id"] and over.definition == "body[5]:id"  # not the signature
assert over.scope.node == "body[5]" and program.entity_of("body[5]") is over  # its scope its implementation's
amb = program.lookup("amb")[0]
assert amb.definition is None and amb.scope is not None
interface = program.lookup("I")[0]
assert interface.definition == "body[6]:id" and interface.scope.kind == "interface"
assert program.lookup("I.a")[0].definition is None  # a signature declares, and defines nothing
assert program.root.separator == "." and program.lookup("N.y")[0].kind == "variable"
assert [e.kind for e in program.lookup("E")] == ["enum"] and program.lookup("E.B")[0].parent.kind == "enum"
assert program.lookup("E.B")[0].definition == "body[11]:body.members[0].id"  # an enum member's place is its id
assert named(program, "two") == ["body[14]:specifiers[1].local"] and named(program, "one") == []  # imported: a name
assert refers(program, "body[2]:declarations[0].init") == []  # `c`, bound nowhere
print(len(listing(program)), "entities")

## TS5DEF-02 · Scopes: `var` in its function, the rest in their blocks; loops, switches, catches and static blocks

In [ ]:
program = define("""
function f(p: number) {
    if (p) { var hoisted = 1; let local = 2; }
    for (let i = 0; i < p; i++) { const j = i; }
    for (const key in o) {}
    for (const value of o) {}
    switch (p) { case 1: let s = 1; }
    try {} catch ({ message }) { message; }
    label: for (;;) { break label; }
    return hoisted + i;
}
class K { static { var inStatic = 1; } }
const fe = function named() { return named; };
const arrow = (a = b, { c } = a) => a + c;
const ce = class Named { m() { return Named; } };
const { [key]: renamed } = o;
const anonymous = class {};
""", ".ts")
assert listing(program) == [
    ("function", "f", 1), ("parameter", "f.p", 1), ("variable", "f.hoisted", 1), ("variable", "local", 1),
    ("variable", "i", 1), ("variable", "j", 1), ("variable", "key", 1), ("variable", "value", 1),
    ("variable", "s", 1), ("variable", "message", 1), ("class", "K", 1), ("variable", "inStatic", 1),
    ("variable", "fe", 1), ("function", "named", 1), ("variable", "arrow", 1), ("parameter", "a", 1),
    ("parameter", "c", 1), ("variable", "ce", 1), ("class", "Named", 1), ("method", "Named.m", 1),
    ("variable", "renamed", 1), ("variable", "anonymous", 1)]
key = named(program, "key")
assert key == ["body[0].body.body[2].left:declarations[0].id", "body[5]:declarations[0].id.properties[0].key"]
assert program.scope_of(key[1]) is program.root and refers(program, key[1]) == []  # the pattern's computed key
scopes = {e.name: e.parent.kind for e in program.entities()}
assert program.lookup("Named") == [] and scopes["Named"] == "block"  # a class expression's name is its own
assert (scopes["local"], scopes["i"], scopes["s"], scopes["message"], scopes["inStatic"]) == (
    "block", "block", "block", "block", "static block")
hoisted, i = named(program, "hoisted")[-1], named(program, "i")[-1]
assert hoisted == "body[0].body.body[7]:argument.left" and refers(program, hoisted) == ["variable:f.hoisted"]
assert program.lookup("f.hoisted")[0].definition == "body[0].body.body[0].consequent.body[0]:declarations[0].id"
assert refers(program, i) == []  # `i` is the loop's
assert program.anchor("body[0].body.body[1].init") is not None  # a `for`'s declaration is a statement
assert refers(program, named(program, "named")[-1]) == ["function:named"]
assert refers(program, named(program, "Named")[-1]) == ["class:Named"]
assert refers(program, named(program, "b")[0]) == [] and refers(program, named(program, "a")[-1]) == ["parameter:a"]
assert named(program, "label") == [] and program.anchor("body[0].body.body[6]:label") is None  # a label names none
assert program.scope_of("body[0].body.body[6].body.body.body[0]").kind == "block"  # the `break`, in the loop's block
assert refers(program, named(program, "message")[-1]) == ["variable:message"]
assert program.scope_of("body[0].body.body[5].handler.body") is program.scope_of("body[0].body.body[5].handler:param")
print("ok")

## TS5DEF-03 · Classes and interfaces: members, accessors, private names, parameter properties and heritage

In [ ]:
program = define("""
abstract class Box<T> extends Base<T> implements Shape {
    #secret = 1;
    static count = 0;
    accessor size = 1;
    [computed] = 2;
    'quoted' = 3;
    "double" = 4;
    0 = 5;
    constructor(private readonly value: T, public label = "x", plain: number) { super(); }
    get area(): number { return this.#secret; }
    set area(v: number) {}
    abstract run(): void;
    method?(): void;
    method?(): void {}
    [key: string]: unknown;
    @decorated field = Box.count;
}
interface Shape<T = number> extends Base<T> {
    area: number; move(): void; get size(): number; [k: string]: unknown; [Symbol.iterator](): void
}
""", ".ts")
box = program.lookup("Box")[0]
assert [(e.kind, e.name, len(e.declarations)) for e in box.scope.entities()] == [
    ("property", "#secret", 1), ("property", "count", 1), ("accessor", "size", 1), ("property", "quoted", 1),
    ("property", "double", 1), ("property", "0", 1), ("method", "constructor", 1), ("property", "value", 1),
    ("property", "label", 1), ("accessor", "area", 2), ("method", "run", 1), ("method", "method", 2),
    ("property", "field", 1)]
assert box.scope.parent.kind == "type parameters"
assert named(program, "T")[1] == "body[0]:superTypeArguments.params[0].typeName"
assert refers(program, named(program, "T")[1]) == ["type parameter:T"]  # `Base<T>`
assert refers(program, named(program, "Base")[0]) == [] and refers(program, named(program, "Shape")[0]) == [
    "interface:Shape"]
assert refers(program, named(program, "computed")[0]) == []
assert named(program, "count") == ["body[0].body.body[1]:key"]  # `Box.count`'s `count` names a property: not visited
assert refers(program, named(program, "Box")[1]) == ["class:Box"]
assert program.entity_of("body[0].body.body[1]") is program.lookup("Box.count")[0]  # the member declares it
constructor = program.lookup("Box.constructor")[0]
assert [e.kind for e in constructor.scope.entities()] == ["parameter", "parameter", "parameter"]
assert constructor.scope.node == "body[0].body.body[7]:value"
assert program.lookup("Box.value")[0].definition == "body[0].body.body[7]:value.params[0]"  # the parameter property
assert program.lookup("Box.constructor.value")[0].definition == "body[0].body.body[7]:value.params[0].parameter"
method = program.lookup("Box.method")[0]
assert method.declarations == ["body[0].body.body[11]:key", "body[0].body.body[12]:key"]
assert method.definition == "body[0].body.body[12]:key" and program.lookup("Box.run")[0].definition is None
assert program.scope_of("body[0].body.body[13]:parameters[0]") is None  # an index signature's parameter names nothing
assert program.scope_of("body[0].body.body[13]:parameters[0].typeAnnotation") is not None
assert named(program, "#secret") == ["body[0].body.body[0]:key"] and named(program, "secret") == []
shape = program.lookup("Shape")[0]
assert [(e.kind, e.name) for e in shape.scope.entities()] == [("property", "area"), ("method", "move"),
                                                             ("accessor", "size")]
assert refers(program, named(program, "Symbol")[0]) == [] and program.lookup("Box.count")[0].kind == "property"
print("ok")

## TS5DEF-04 · Meanings: a name is a value, a type or a namespace where it is written, and finds only those

In [ ]:
program = define("""
type Point = { x: number };
const Point = 1;
namespace Geo { export interface Shape {} }
let p: Point = Point;
let q: typeof Point;
let s: Geo.Shape;
let g = Geo;
let a = <Point>p, b = p as Point, c = p satisfies Point, d = [p] as const;
function guard(x: unknown): x is Point { return true; }
export { Point };
export { Point as Other } from "m";
const Point2 = Point;
""", ".ts")
point = program.named("Point")
assert [program.space_of(at) for at in point] == [None, None, "type", "value", "value", "type", "type", "type",
                                                  "type", None, "value"]
assert refers(program, point[2]) == ["type alias:Point"] and refers(program, point[3]) == ["variable:Point"]
assert refers(program, point[4]) == ["variable:Point"]  # `typeof Point`
assert refers(program, point[9]) == ["type alias:Point", "variable:Point"]  # an export names whatever it means
assert point[9] == "body[9]:specifiers[0].local" and program.scope_of("body[10]:specifiers[0].local") is None
geo = program.named("Geo")
assert [program.space_of(at) for at in geo] == [None, "namespace", "value"]
assert refers(program, geo[1]) == ["namespace:Geo"]
assert named(program, "Shape") == ["body[2].body.body[0].declaration:id"]  # `Geo.Shape`'s `Shape`: not visited
assert refers(program, named(program, "x")[1]) == ["parameter:guard.x"]
assert named(program, "const") == [] and program.scope_of("body[7]:declarations[3].init.typeAnnotation") is None
assert D.MEANINGS["class"] == {"value", "type", "namespace"} and D.MEANINGS["interface"] == {"type"}
print("ok")

## TS5DEF-05 · Namespaces, ambient modules, `declare global` and `import a = b.c`

In [ ]:
program = define("""
namespace Outer.Inner { export const deep = 1; }
namespace Outer { export namespace Inner { export const also = 2; } }
declare module "virtual" { export const v: number; }
declare module "short";
declare module "virtual" { export const w: number; }
declare global { interface Window { app: string } }
import Alias = Outer.Inner;
import Missing = Nowhere.Thing;
import Self = Self.X;
import Required = require("m");
import Single = Outer;
const use = Alias.deep;
""", ".ts")
assert listing(program)[:6] == [
    ("namespace", "Outer", 2), ("namespace", "Outer.Inner", 2), ("variable", "Outer.Inner.deep", 1),
    ("variable", "Outer.Inner.also", 1), ("module", "virtual", 2), ("variable", "virtual.v", 1)]
assert program.lookup("Outer.Inner")[0].declarations == ["body[0]:id.right", "body[1].body.body[0].declaration:id"]
assert program.entity_of("body[0]") is program.lookup("Outer.Inner")[0]  # `namespace a.b` defines `a.b`
assert [e.kind for e in program.lookup("short")] == ["module"] and program.lookup("short")[0].scope.kind == "module"
assert program.lookup("Window")[0].kind == "interface" and named(program, "global") == []
alias, missing, self_, required, single = (program.lookup(name)[0] for name in (
    "Alias", "Missing", "Self", "Required", "Single"))
assert alias.target.qualified_name() == "Outer.Inner" and alias.resolved().kind == "namespace"
assert single.target is program.lookup("Outer")[0]
assert missing.target is None and self_.target is None and required.target is None
assert refers(program, named(program, "Alias")[1]) == ["import:Alias"]
assert program.space_of("body[6]:moduleReference.left") == "namespace"
print("ok")

## TS5DEF-06 · Types' own scopes: signatures, mapped types, `infer`, and JSX's components

In [ ]:
program = define("""
enum Flags { A, // a comment
  [computed] = 1 }
type Mapped<T> = { [K in keyof T]: T[K] };
type Element<T> = T extends (infer E extends string)[] ? E : never;
type Fn = <U>(arg: U, ...rest: U[]) => U;
type Ctor = new (x: number) => object;
type Lit = { (a: string): void; new (b: string): Lit; m<V>(c: V): V; [k: string]: unknown };
""", ".ts")
assert [(e.kind, e.name, e.parent.kind) for e in program.entities()] == [
    ("enum", "Flags", "module"), ("enum member", "A", "enum"), ("type alias", "Mapped", "module"),
    ("type parameter", "T", "type parameters"),
    ("type parameter", "K", "type parameters"), ("type alias", "Element", "module"),
    ("type parameter", "T", "type parameters"), ("type parameter", "E", "type parameters"),
    ("type alias", "Fn", "module"), ("type parameter", "U", "function"), ("parameter", "arg", "function"),
    ("parameter", "rest", "function"), ("type alias", "Ctor", "module"), ("parameter", "x", "function"),
    ("type alias", "Lit", "module"), ("parameter", "a", "function"), ("parameter", "b", "function"),
    ("type parameter", "V", "function"), ("parameter", "c", "function")]
assert program.anchor("body[0]:body.members[1]").statement == "body[0]:body.members[1]"  # the comment, a statement
assert refers(program, named(program, "computed")[0]) == []
assert refers(program, named(program, "V")[-1]) == ["type parameter:V"]
assert named(program, "K") == ["body[1]:typeAnnotation.key", "body[1]:typeAnnotation.body.indexType.typeName"]
assert refers(program, named(program, "K")[1]) == ["type parameter:K"]
assert refers(program, named(program, "E")[1]) == ["type parameter:E"]
assert refers(program, named(program, "U")[-1]) == ["type parameter:U"]
assert refers(program, named(program, "Lit")[1]) == ["type alias:Lit"]
assert named(program, "k") == []  # an index signature's parameter names nothing
program = define("""
function Component() {}
const ns = { Item: Component };
const view = <div><Component /><ns.Item>text</ns.Item><svg:path d={ns} /></div>;
""", ".tsx")
component = named(program, "Component")
assert component[2] == "body[2]:declarations[0].init.children[0].openingElement.name"
assert refers(program, component[2]) == ["function:Component"] and program.space_of(component[2]) == "value"
assert named(program, "ns")[1:3] == ["body[2]:declarations[0].init.children[1].openingElement.name.object",
                                     "body[2]:declarations[0].init.children[1].closingElement.name.object"]
assert [refers(program, at) for at in named(program, "ns")[1:]] == [["variable:ns"]] * 3
assert named(program, "Item") == [] and named(program, "div") == [] and named(program, "svg") == []
print("ok")

## TS5DEF-07 · Statements within terms: function expressions, arrows' blocks, class expressions and methods

A term that holds statements holds them as its property, a function expression's or a block-bodied arrow's `block`, a
class expression's `classBody`: each is anchored at the term's place, followed by the statement places within, to any
depth, and its names bind in the scope the term opens.

In [ ]:
program = define("""
const outer = function () {
    if (outer) { var hoisted = 1; }
    const inner = () => {
        let deep = hoisted;
        return class Local extends Base {
            field = deep;
            method() { const own = Local; return function named() { return named; }; }
            static { var s = own; }
        };
    };
};
const object = { method() { let m = 1; return m; }, get g() { return outer; } };
const short = (x) => x;
""", ".ts")
assert listing(program) == [
    ("variable", "outer", 1), ("variable", "hoisted", 1), ("variable", "inner", 1), ("variable", "deep", 1),
    ("class", "Local", 1), ("property", "Local.field", 1), ("method", "Local.method", 1),
    ("variable", "Local.method.own", 1), ("function", "named", 1), ("variable", "s", 1), ("variable", "object", 1),
    ("variable", "m", 1), ("variable", "short", 1), ("parameter", "x", 1)]
function = "body[0]:declarations[0].init"
arrow = f"{function}.block.body[1]:declarations[0].init"
local = f"{arrow}.block.body[1]:argument"
assert program.anchor(f"{function}.block").statement == f"{function}.block"  # the block, at the term's place
assert program.scope_of(f"{function}.block") is program.scope_of(f"{function}.block.body[0]")
block = program.scope_of(f"{function}.block")
assert block.kind == "function" and block.node == function
hoisted = D.referents(program, f"{arrow}.block.body[0]:declarations[0].init")[0]  # `var`, in the function's scope
assert hoisted.definition == f"{function}.block.body[0].consequent.body[0]:declarations[0].id"
assert hoisted.parent is program.scope_of(f"{function}.block")
assert refers(program, f"{arrow}.block.body[0]:declarations[0].init") == ["variable:hoisted"]
deep = D.referents(program, f"{local}.classBody.body[0]:value")[0]  # a field's value sees the arrow's names
assert deep.definition == f"{arrow}.block.body[0]:declarations[0].id"
local_class = D.referents(program, f"{local}.id")[0]
assert local_class.kind == "class" and program.entity_of(local) is local_class
assert local_class.scope.node == local and program.scope_of(f"{local}.classBody") is local_class.scope
own = f"{local}.classBody.body[1]:value.block.body[0]"
assert D.referents(program, f"{own}:declarations[0].id")[0].qualified_name() == "Local.method.own"
assert D.referents(program, f"{own}:declarations[0].id")[0].definition == f"{own}:declarations[0].id"
assert refers(program, f"{own}:declarations[0].init") == ["class:Local"]
named_function = f"{local}.classBody.body[1]:value.block.body[1]:argument"
assert refers(program, f"{named_function}.block.body[0]:argument") == ["function:named"]
assert program.entity_of(named_function) is D.referents(program, f"{named_function}.id")[0]
assert refers(program, f"{local}.classBody.body[2].body[0]:declarations[0].init") == []  # a method's own, unseen
assert D.referents(program, f"{local}.classBody.body[2].body[0]:declarations[0].id")[0].parent.kind == "static block"
assert refers(program, "body[1]:declarations[0].init.properties[0].value.block.body[1]:argument") == ["variable:m"]
assert refers(program, "body[1]:declarations[0].init.properties[1].value.block.body[0]:argument") == [
    "variable:outer"]
assert refers(program, "body[2]:declarations[0].init.body") == ["parameter:x"]  # an arrow's expression
declarations = [d for e in program.entities() for d in e.declarations]
assert sorted(declarations, key=lambda at: at.order) == declarations  # in source order, through every block
print("ok")

## TS5DEF-08 · Every entity of the conformance sources, every name located, and what each pattern binds

Read independently of `define`, by a walk of the stored program's value: every statement, clause, class member and
error, a class's body and a namespace's block, and every block or class body a term holds, through `Terms.kind_of` and
`Objects.properties`, is located, its anchor's text its place in the program's value, and the anchors come in the
walk's order, which is source order. Every other place is located, unless it is where a name names nothing (a property's
name, a label, an import's imported name, ...): the walk does not descend there, and the pairs of kind and property
where it stops are listed. Every identifier located has its name written there (but a `this` parameter's), and each
declarator, parameter and catch clause binds the names mbse-expressions' `bound_names` gives.

In [ ]:
HOLDERS = {f"{Statements.LANGUAGE}.{name}" for name in (*Statements.KINDS, "Error", "ClassBody", "TSModuleBlock")}
LISTS = {f"{Statements.LANGUAGE}.{name}" for name in ("Body", "Cases", "ClassElements")}
BINDS = {("variabledeclarator", "id"), ("catchclause", "param")}  # and every `params`, but TYPE_PARAMETERS'
TYPE_PARAMETERS = ("tstypeparameterdeclaration", "tstypeparameterinstantiation")


def record_of(value):
    """The record a value holds through unions, and its kind: a term's, or its schema's name in lower case."""
    while not Terms.is_term(value) and isinstance(value.as_schema(), Schemas.OfUnion.Data):
        ((_, value),) = Objects.properties(value).items()
    return Terms.kind_of(value) if Terms.is_term(value) else (value.as_schema().name.rsplit(".", 1)[-1].lower(), value)


def holds(record, name, value):
    """Whether a property holds statement places: statements, clauses, members, errors, a body or a block."""
    if record.as_schema().properties[name].schema.name in LISTS:
        return True
    return hasattr(value, "as_schema") and not Terms.is_term(value) and record_of(value)[1].as_schema().name in HOLDERS


def conformance(source):
    """The program of a conformance source, every place of it checked; the walk's counts, and where it stops."""
    STORE.transaction(Parse.parse((SOURCES / source).read_text(encoding="utf-8"), source))
    started = time.perf_counter()
    program = D.define(STORE, source)
    elapsed = time.perf_counter() - started
    declared = [str(at) for e in program.entities() for at in e.declarations]
    anchors, unlocated, counts = [], set(), {"identifiers": 0, "patterns": 0}

    def bound(place):
        return {program.name_at(at) for at in declared if at == place or at.startswith((f"{place}.", f"{place}["))}

    def walk(record, path):
        anchor = program.anchor(path or "<program>")
        assert program.scope_of(anchor) is not None and anchor.property is None and anchor.statement == path, path
        anchors.append(anchor)
        for name, value in Objects.properties(record).items():
            if not holds(record, name, value):
                parts(value, f"{path}:{name}", (record_of(record)[0], name))
                continue
            for i, element in enumerate(value.values) if isinstance(value, Objects.Listed) else [(None, value)]:
                step = name if i is None else f"{name}[{i}]"
                walk(record_of(element)[1], f"{path}.{step}" if path else step)

    def parts(value, place, context):
        if isinstance(value, Objects.Listed):
            for i, element in enumerate(value.values):
                parts(element, f"{place}[{i}]", context)
            return
        if not hasattr(value, "as_schema"):
            return
        tag, record = record_of(value)
        if record.as_schema().name in HOLDERS:  # a block or a class body a term holds
            walk(record, place)
            return
        if program.scope_of(place) is None:
            unlocated.add(context)
            return
        if tag == "identifier":
            counts["identifiers"] += 1
            name = Objects.properties(record)["name"]
            assert program.name_at(place) == (None if name == "this" else name), place  # `this: T` binds nothing
        if context in BINDS or context[1] == "params" and context[0] not in TYPE_PARAMETERS:
            counts["patterns"] += 1
            assert bound(place) == TypeScriptExpressions.bound_names(value) - {"this"}, place
        for name, held in Objects.properties(record).items():
            parts(held, f"{place}.{name}", (tag, name))

    with STORE.transaction():
        walk(STORE.singleton(WORKSPACE).files({"path": source}), "")
    assert program.anchor("<program>") == program.root.node == "<program>"
    assert [a.order for a in anchors] == sorted(a.order for a in anchors)  # the walk's order is source order
    return program, len(anchors), counts, unlocated, elapsed


program, statements, counts, unlocated, elapsed = conformance("typescript59.ts")
kinds = {}
for entity in program.entities():
    kinds[entity.kind] = kinds.get(entity.kind, 0) + 1
assert set(kinds) == set(D.MEANINGS), set(D.MEANINGS) - set(kinds)
assert (statements, counts) == (208, {"identifiers": 271, "patterns": 70}), (statements, counts)
assert unlocated == {
    ("breakstatement", "label"), ("continuestatement", "label"), ("labeledstatement", "label"),
    ("importdeclaration", "source"), ("importdeclaration", "attributes"), ("importspecifier", "imported"),
    ("exportalldeclaration", "exported"), ("exportalldeclaration", "source"), ("exportspecifier", "exported"),
    ("memberexpression", "property"), ("metaproperty", "meta"), ("metaproperty", "property"), ("property", "key"),
    ("tspropertysignature", "key"), ("tsmethodsignature", "key"), ("tsqualifiedname", "right"),
    ("tsnamedtuplemember", "label"), ("tsindexsignature", "parameters"), ("tsimporttype", "source"),
    ("tsimporttype", "qualifier"), ("tsasexpression", "typeAnnotation"),  # `as const`
    ("tsmoduledeclaration", "id")}, unlocated  # `declare global`
point2 = program.lookup("Point2")[0]
assert point2.kind == "import" and point2.target.qualified_name() == "Geometry.Point"
assert program.lookup("Container.value")[0].kind == "property"  # a parameter property
assert program.lookup("Shape.size")[0].declarations == ["body[78]:body.body[3].key", "body[78]:body.body[4].key"]
print(sum(kinds.values()), "entities;", statements, "statement places,", counts, f"defined in {elapsed:.2f} s")
program, statements, counts, unlocated, elapsed = conformance("typescript59-jsx.tsx")
assert (statements, counts) == (9, {"identifiers": 13, "patterns": 4}), (statements, counts)
assert unlocated == {("importdeclaration", "source"), ("memberexpression", "property"), ("property", "key"),
                     ("tspropertysignature", "key"), ("jsxopeningelement", "name"), ("jsxclosingelement", "name"),
                     ("jsxattribute", "name")}, unlocated
assert [refers(program, at) for at in program.named("React")] == [["import:React"]] * 3
assert refers(program, named(program, "Component")[0]) == ["function:Component"]
print(len(listing(program)), "entities;", statements, "statement places,", counts, f"defined in {elapsed:.2f} s")

## TS5DEF-09 · Anchors, and the transaction `define` reads in

An anchor is plain data: its text, its parts and its order, read with no transaction open; the text of a statement's
anchor is its place in the program's value. `define` joins no transaction: it opens its own, and `define_file` reads
within the caller's; a path the workspace holds no file at is a LookupError.

In [ ]:
program = define("""
function f(a) {
    return a;
}
const g = () => { a; };
""", ".ts")
use = program.named("a")[1]
assert (use.statement, use.property, use.term) == ("body[0].body.body[0]", "argument", ()) and str(use) == (
    "body[0].body.body[0]:argument")
assert repr(use) == "Anchor('body[0].body.body[0]:argument')" and use == program.anchor(str(use)) and use != 1
assert hash(use) == hash("body[0].body.body[0]:argument") and use in {"body[0].body.body[0]:argument"}
parameter = program.named("a")[0]
assert (parameter.property, parameter.term) == ("params", (0,)) and parameter.order < use.order
within = program.named("a")[2]  # within an arrow's block
assert (within.statement, within.property, within.term) == ("body[1]:declarations[0].init.block.body[0]",
                                                            "expression", ())
assert program.root.node == "<program>" and program.anchor("body[0].body").statement == "body[0].body"
with STORE.transaction():
    file = STORE.singleton(WORKSPACE).files({"path": program.path})
    function = Objects.properties(file)["body"].values[0]  # body[0]
    kind = Terms.kind_of(function)[0]
    joined = D.define_file(STORE, file)  # within the caller's transaction, from the program's visitor
    with raises(ValueError, match="transactions do not nest"):  # define opens its own: called with none open
        D.define(STORE, program.path)
assert kind == "functiondeclaration" and program.scope_of("body[0].body.body[0]") is program.lookup("f")[0].scope
assert listing(joined) == listing(program) == [("function", "f", 1), ("parameter", "f.a", 1), ("variable", "g", 1)]
assert D.referents(joined, use) == [joined.lookup("f.a")[0]]  # an anchor of one program finds its place in another
assert program.lookup("a", at=use) == program.lookup("f.a") and refers(program, within) == []
with raises(LookupError, match="absent.ts"):
    D.define(STORE, "absent.ts")
assert D.referents(program, "body[9]:argument") == [] and program.anchor("body[9]:argument") is None
print("ok")

## TS5DEF-10 · Invalid syntax: an `Error` binds nothing itself, the statements recovered within it bind in its scope, wherever it stands; an `error` term's recovered terms are uses, and a `missing` term binds nothing

In [ ]:
program = define("""import os from "os";
if (x { let inner = os; }
class C { m() { return os; } 5x; }
const y = (z + );
call(a, , b);
""", ".ts")
assert listing(program) == [("import", "os", 1), ("variable", "inner", 1), ("class", "C", 1), ("method", "C.m", 1),
                            ("property", "C.5", 1), ("variable", "y", 1)]
assert program.anchor("body[1]") is not None and program.scope_of("body[1]") is program.root
assert program.lookup("inner")[0].definition == "body[1].body[0]:declarations[0].id"  # recovered, in its scope
assert refers(program, "body[1].body[0]:declarations[0].init") == ["import:os"]
member = program.anchor("body[2].body.body[2]")  # an `Error` where a member stands
assert member is not None and program.scope_of(member) is program.lookup("C")[0].scope
assert named(program, "z") == ["body[3]:declarations[0].init.expression.left"]
assert program.anchor("body[3]:declarations[0].init.expression.right") is not None  # `missing`: located, no name
assert program.name_at("body[3]:declarations[0].init.expression.right") is None
assert named(program, "a") == ["body[4]:expression.terms[1]"] and refers(program, "body[4]:expression.terms[1]") == []
print("ok")